# Chapter 6, Guide 2: Quantum counting

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 6,
"Building the counting circuit" and the warning box "The diffuser's sign, and reading
the output".

Steps, in the book's order: the qubit budget, controlled-$G^{2^k}$ by repetition,
controlling only the two multi-controlled gates, the $Z$ on the control that fixes the
diffuser's sign, reading $\mu$ off the measured $m$, and the two tests (eigenvalues of
$G$, then the histogram).

Convention: book wire $j$ (top, most significant) is Qiskit qubit `bq(j)`; see the
Grover notebook.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import QFTGate
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

np.set_printoptions(precision=4, suppress=True)
bq = lambda j, tot: tot - 1 - j

## The Grover operator, with and without a control

The oracle marks a *set* of targets this time, so $\mu$ can be 1 or 2. Only the
oracle's $C^n(X)$ and the diffuser's central $C^{n-1}(X)$ receive the control; the
Hadamard and $X$ layers cancel on their own when the control is off.

In [2]:
def oracle(n, targets, ctrl=None):
    """Bit-flip oracle onto the ancilla for a set of targets; optional extra control."""
    tot = n + 1 + (0 if ctrl is None else 1)
    qc = QuantumCircuit(tot)
    search = [bq(j, tot) for j in range(n)]
    anc = bq(n, tot)
    for w in targets:
        bits = [(w >> (n - 1 - j)) & 1 for j in range(n)]
        zeros = [search[j] for j, b in enumerate(bits) if b == 0]
        if zeros: qc.x(zeros)
        qc.mcx(search + ([] if ctrl is None else [ctrl]), anc)
        if zeros: qc.x(zeros)
    return qc


def diffuser_ctrl(n, tot, ctrl=None, fix_sign=False):
    """-V = H^n X^n C^{n-1}(Z) X^n H^n on book wires 0..n-1 of a tot-qubit circuit.
    With ctrl: control the central gate; with fix_sign: Z on ctrl so that C(-V) -> C(V)."""
    qc = QuantumCircuit(tot)
    search = [bq(j, tot) for j in range(n)]
    t = search[-1]
    qc.h(search); qc.x(search); qc.h(t)
    qc.mcx(search[:-1] + ([] if ctrl is None else [ctrl]), t)
    qc.h(t); qc.x(search); qc.h(search)
    if fix_sign and ctrl is not None:
        qc.z(ctrl)
    return qc


def grover_op_matrix(n, targets):
    """G = (-V) U_f restricted to the search register, as a 2^n x 2^n matrix."""
    N = 2**n
    Uf = np.eye(N); Uf[targets, targets] = -1
    s = np.ones(N) / np.sqrt(N)
    V = 2 * np.outer(s, s) - np.eye(N)
    return V @ Uf          # the true G (the circuit realizes -G; see the sign box)

## Test 1: eigenvalues of $G$ are $e^{\pm 2i\theta}$, $\theta=\arcsin\sqrt{\mu/N}$

In [3]:
n = 3; N = 2**n
for targets in ([5], [2, 5]):
    mu = len(targets)
    theta = np.arcsin(np.sqrt(mu / N))
    ev = np.linalg.eigvals(grover_op_matrix(n, targets))
    # outside the 2D rotation plane G acts as ±1 (-1 on non-targets orthogonal to |s>,
    # +1 on target combinations orthogonal to the uniform one); drop those
    nontrivial = ev[(np.abs(ev - 1) > 1e-9) & (np.abs(ev + 1) > 1e-9)]
    phases = np.sort(np.angle(nontrivial))
    print(f"mu={mu}: eigenphases {phases}, expected ±2theta = ±{2*theta:.4f}")
    assert np.allclose(np.abs(phases), 2 * theta)
print("Test 1 passed")

mu=1: eigenphases [-0.7227  0.7227], expected ±2theta = ±0.7227
mu=2: eigenphases [-1.0472  1.0472], expected ±2theta = ±1.0472
Test 1 passed


## The counting circuit

$c$ counting qubits (book wires $0..c-1$, wire 0 most significant), then the $n$ search
wires, then the oracle ancilla. Counting wire $k$ controls $G^{2^{c-1-k}}$, done by
$2^{c-1-k}$ controlled repetitions, because $G$ is a circuit and not a classical
constant. The inverse QFT follows, with the most significant bit on top (`QFTGate` takes its
qubit list least-significant first, hence the reversed list).

In [4]:
def counting_circuit(n, c, targets, fix_sign=True):
    tot = c + n + 1
    qc = QuantumCircuit(tot, c)
    count = [bq(k, tot) for k in range(c)]
    search = [bq(c + j, tot) for j in range(n)]
    anc = bq(c + n, tot)
    qc.x(anc); qc.h(anc)
    qc.h(count); qc.h(search)
    for k in range(c):
        reps = 2 ** (c - 1 - k)
        orc = oracle(n, targets, ctrl=0)                     # built on its own wire set
        dif = diffuser_ctrl(n, n + 1 + 1, ctrl=0, fix_sign=fix_sign)
        # the sub-circuits use Qiskit qubit 0 as control, then search wires, then ancilla
        # map: sub-qubit 0 -> count[k]; sub search wires -> search; sub ancilla -> anc
        # In the sub-circuits the search wires are bq(j, n+2) for j in range(n) = n+1-j,
        # and the ancilla is bq(n, n+2) = 1, control is 0. Build the qubit map accordingly.
        qmap = [None] * (n + 2)
        qmap[0] = count[k]
        for j in range(n):
            qmap[n + 1 - j] = search[j]
        qmap[1] = anc
        for _ in range(reps):
            qc.compose(orc, qubits=qmap, inplace=True)
            qc.compose(dif, qubits=qmap, inplace=True)
    # inverse QFT on the counting register; Qiskit's QFT takes its qubit list LSB first
    qc.append(QFTGate(c).inverse(), count[::-1])
    qc.measure(count, [c - 1 - k for k in range(c)])           # classical bit order = book order
    return qc


def mu_from_m(m, c, N):
    two_theta = 2 * np.pi * m / 2**c
    theta = two_theta / 2
    return N * np.sin(theta) ** 2


c = 6
qc = counting_circuit(n, c, [5])
print("qubits:", qc.num_qubits, " Grover iterations inside:", 2**c - 1)

qubits: 10  Grover iterations inside: 63


## Test 2: the histogram peaks at the two predicted values of $m$

Predicted: $m \approx 2^c \cdot 2\theta/2\pi$ and its mirror $2^c - m$ (the
$e^{-2i\theta}$ branch), both of which give the same $\mu$.

In [5]:
sim = AerSimulator()
for targets in ([5], [2, 5]):
    mu = len(targets); theta = np.arcsin(np.sqrt(mu / N))
    m_pred = 2**c * (2 * theta) / (2 * np.pi)
    qc = counting_circuit(n, c, targets, fix_sign=True)
    counts = sim.run(transpile(qc, sim), shots=4000).result().get_counts()
    top = sorted(counts.items(), key=lambda kv: -kv[1])[:4]
    print(f"\nmu = {mu}: predicted m = {m_pred:.2f} or {2**c - m_pred:.2f}")
    for s_, cnt in top:
        m = int(s_, 2)
        print(f"  m={m:2d} ({cnt:4d} shots) -> mu estimate {mu_from_m(m, c, N):.3f} -> rounds to {round(mu_from_m(m, c, N))}")
    best_m = int(top[0][0], 2)
    assert round(mu_from_m(best_m, c, N)) == mu
print("\nTest 2 passed: the most frequent m rounds to the correct mu")


mu = 1: predicted m = 7.36 or 56.64
  m=57 (1328 shots) -> mu estimate 0.908 -> rounds to 1
  m= 7 (1248 shots) -> mu estimate 0.908 -> rounds to 1
  m=56 ( 415 shots) -> mu estimate 1.172 -> rounds to 1
  m= 8 ( 382 shots) -> mu estimate 1.172 -> rounds to 1



mu = 2: predicted m = 10.67 or 53.33
  m=53 (1395 shots) -> mu estimate 2.114 -> rounds to 2
  m=11 (1306 shots) -> mu estimate 2.114 -> rounds to 2
  m=10 ( 348 shots) -> mu estimate 1.778 -> rounds to 2
  m=54 ( 334 shots) -> mu estimate 1.778 -> rounds to 2

Test 2 passed: the most frequent m rounds to the correct mu


## The sign trap, reproduced

Drop the $Z$ on the control and the estimate flips to the complement $N-\mu$: the book
quotes $7$ for $\mu=1$ and $6$ for $\mu=2$ at $n=3$, $c=6$.

In [6]:
for targets in ([5], [2, 5]):
    qc = counting_circuit(n, c, targets, fix_sign=False)
    counts = sim.run(transpile(qc, sim), shots=4000).result().get_counts()
    best_m = int(max(counts, key=counts.get), 2)
    print(f"mu={len(targets)} without the Z fix: estimate {round(mu_from_m(best_m, c, N))}  (= N - mu)")

mu=1 without the Z fix: estimate 7  (= N - mu)


mu=2 without the Z fix: estimate 6  (= N - mu)


## Resolution against $c$

$\delta\mu \approx \sqrt{\mu(N-\mu)}\,2\pi/2^c$: each counting qubit halves the error.

In [7]:
for cc in [4, 5, 6, 7]:
    print(f"c={cc}: delta_mu ≈ {np.sqrt(1*(N-1))*2*np.pi/2**cc:.3f} for mu=1, N={N};  oracle queries = {2**cc-1}")

c=4: delta_mu ≈ 1.039 for mu=1, N=8;  oracle queries = 15
c=5: delta_mu ≈ 0.519 for mu=1, N=8;  oracle queries = 31
c=6: delta_mu ≈ 0.260 for mu=1, N=8;  oracle queries = 63
c=7: delta_mu ≈ 0.130 for mu=1, N=8;  oracle queries = 127
